In [0]:
silver_drivers = spark.table(
    "workspace.transportation_analytics.silver_drivers"
)

silver_driver_metrics = spark.table(
    "workspace.transportation_analytics.silver_driver_monthly_metrics"
)

print("Silver driver tables loaded successfully")

silver_drivers.printSchema()
silver_driver_metrics.printSchema()

Silver driver tables loaded successfully
root
 |-- driver_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- termination_date: date (nullable = true)
 |-- license_number: string (nullable = true)
 |-- license_state: string (nullable = true)
 |-- date_of_birth: date (nullable = true)
 |-- home_terminal: string (nullable = true)
 |-- employment_status: string (nullable = true)
 |-- cdl_class: string (nullable = true)
 |-- years_experience: integer (nullable = true)

root
 |-- driver_id: string (nullable = true)
 |-- month: date (nullable = true)
 |-- trips_completed: integer (nullable = true)
 |-- total_miles: integer (nullable = true)
 |-- total_revenue: double (nullable = true)
 |-- average_mpg: double (nullable = true)
 |-- total_fuel_gallons: double (nullable = true)
 |-- on_time_delivery_rate: double (nullable = true)
 |-- average_idle_hours: double (nullable = true)



In [0]:
from pyspark.sql import functions as F

In [0]:
gold_driver_performance = (
    silver_driver_metrics
    .join(
        silver_drivers,
        on="driver_id",
        how="left"
    )
)

print("Driver data joined successfully")

Driver data joined successfully


In [0]:
gold_driver_performance = (
    gold_driver_performance
    .groupBy(
        "driver_id",
        "first_name",
        "last_name",
        "employment_status",
        "years_experience"
    )
    .agg(
        F.sum("trips_completed").alias("total_trips"),
        F.sum("total_miles").alias("total_miles"),
        F.sum("total_revenue").alias("total_revenue"),
        F.avg("on_time_delivery_rate").alias("avg_on_time_delivery_rate"),
        F.sum("total_miles").alias("miles_for_mpg"),
        F.sum("total_fuel_gallons").alias("total_fuel_gallons"),
        F.avg("average_idle_hours").alias("avg_idle_hours")
    )
)

print("Driver performance calculated successfully")

Driver performance calculated successfully


In [0]:
gold_driver_performance = (
    gold_driver_performance
    .withColumn(
        "overall_mpg",
        F.round(
            F.col("miles_for_mpg") / F.col("total_fuel_gallons"),
            2
        )
    )
    .withColumn(
        "revenue_per_mile",
        F.round(
            F.col("total_revenue") / F.col("total_miles"),
            2
        )
)
)

print("MPG and revenue per mile calculated successfully")

MPG and revenue per mile calculated successfully


In [0]:
from delta.tables import DeltaTable

target = DeltaTable.forName(
    spark,
    "workspace.transportation_analytics.gold_driver_performance"
)

target.alias("t").merge(
    gold_driver_performance.alias("s"),
    "t.driver_id = s.driver_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

print("Gold Driver Performance table updated using MERGE")

Gold Driver Performance table updated using MERGE


In [0]:
spark.table(
    "workspace.transportation_analytics.gold_driver_performance"
).show(20, truncate=False)

+---------+----------+---------+-----------------+----------------+-----------+-----------+------------------+-------------------------+-------------+------------------+------------------+-----------+----------------+
|driver_id|first_name|last_name|employment_status|years_experience|total_trips|total_miles|total_revenue     |avg_on_time_delivery_rate|miles_for_mpg|total_fuel_gallons|avg_idle_hours    |overall_mpg|revenue_per_mile|
+---------+----------+---------+-----------------+----------------+-----------+-----------+------------------+-------------------------+-------------+------------------+------------------+-----------+----------------+
|DRV00009 |Karen     |Hernandez|Active           |13              |669        |966516     |2073798.9400000002|0.443388888888889        |966516       |150888.1          |7.027777777777778 |6.41       |2.15            |
|DRV00010 |Sarah     |Gonzalez |Active           |23              |709        |998116     |2144658.93        |0.4374722222222222

In [0]:
display(
    gold_driver_performance
    .orderBy(F.desc("total_revenue"))
    .select(
        "driver_id",
        "first_name",
        "last_name",
        "total_trips",
        "total_miles",
        "total_revenue",
        "avg_on_time_delivery_rate",
        "overall_mpg",
        "revenue_per_mile"
    )
)

driver_id,first_name,last_name,total_trips,total_miles,total_revenue,avg_on_time_delivery_rate,overall_mpg,revenue_per_mile
DRV00051,Linda,Davis,723,1074677,2311844.3,0.4342222222222223,6.43,2.15
DRV00108,Joseph,Jones,735,1073445,2311327.9000000004,0.4771666666666667,6.46,2.15
DRV00059,Barbara,Gonzalez,729,1064844,2305056.6000000006,0.4596666666666666,6.47,2.16
DRV00016,Thomas,Gonzalez,712,1042631,2278574.88,0.45011111111111124,6.45,2.19
DRV00019,Robert,Jackson,749,1060268,2275654.7600000002,0.4491944444444445,6.42,2.15
DRV00127,Thomas,Gonzalez,702,1047108,2272317.62,0.42055555555555557,6.44,2.17
DRV00149,Linda,Jones,721,1059718,2269105.6699999995,0.45222222222222225,6.46,2.14
DRV00147,Richard,Garcia,705,1039157,2256066.92,0.4469444444444444,6.44,2.17
DRV00085,Linda,Wilson,703,1034233,2229521.23,0.4538888888888889,6.46,2.16
DRV00087,William,Anderson,725,1029613,2214003.33,0.4190555555555555,6.46,2.15


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.